# Task 2: End-to-End ML Pipeline with Scikit-learn Pipeline API
**DevelopersHub Corporation – AI/ML Engineering Internship**

## Problem Statement
Build a reusable, production-ready ML pipeline to predict customer churn using the Telco Churn dataset.

## Objective
- Implement preprocessing (scaling, encoding) via `sklearn.Pipeline`
- Train Logistic Regression and Random Forest models
- Tune hyperparameters using `GridSearchCV`
- Export pipeline using `joblib`

## 1. Install Dependencies

In [ ]:
!pip install scikit-learn pandas numpy matplotlib seaborn joblib -q

## 2. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix,
    roc_curve, precision_recall_curve
)

np.random.seed(42)
print("✅ Libraries imported successfully")

## 3. Dataset Loading

In [ ]:
# Load Telco Churn Dataset
# Option 1: Download directly
import urllib.request
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
urllib.request.urlretrieve(url, 'telco_churn.csv')
df = pd.read_csv('telco_churn.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nData types:\n{df.dtypes.value_counts()}")

In [ ]:
# Preview the data
display(df.head())
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
print(f"\nChurn distribution:\n{df['Churn'].value_counts()}")

## 4. Exploratory Data Analysis

In [ ]:
# Churn rate visualization
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Telco Churn – EDA', fontsize=16, fontweight='bold')

# 1. Churn distribution
churn_counts = df['Churn'].value_counts()
axes[0, 0].pie(churn_counts, labels=['No Churn', 'Churn'],
               colors=['#2ECC71', '#E74C3C'],
               autopct='%1.1f%%', startangle=90,
               wedgeprops=dict(edgecolor='white', linewidth=2))
axes[0, 0].set_title('Churn Distribution')

# 2. Monthly charges vs Churn
df['MonthlyCharges'] = pd.to_numeric(df['MonthlyCharges'], errors='coerce')
df.boxplot(column='MonthlyCharges', by='Churn', ax=axes[0, 1],
           boxprops=dict(color='#3498DB'),
           medianprops=dict(color='#E74C3C', linewidth=2))
axes[0, 1].set_title('Monthly Charges by Churn')
axes[0, 1].set_xlabel('Churn')

# 3. Tenure distribution
df['tenure'] = pd.to_numeric(df['tenure'], errors='coerce')
df.groupby('Churn')['tenure'].plot.hist(alpha=0.6, bins=30, ax=axes[0, 2])
axes[0, 2].set_title('Tenure Distribution by Churn')
axes[0, 2].legend(['No Churn', 'Churn'])

# 4. Contract type vs Churn
ct = df.groupby(['Contract', 'Churn']).size().unstack()
ct.plot(kind='bar', ax=axes[1, 0], color=['#2ECC71', '#E74C3C'], edgecolor='black')
axes[1, 0].set_title('Contract Type vs Churn')
axes[1, 0].set_xlabel('Contract Type')
axes[1, 0].tick_params(axis='x', rotation=30)

# 5. Internet Service vs Churn
it = df.groupby(['InternetService', 'Churn']).size().unstack()
it.plot(kind='bar', ax=axes[1, 1], color=['#2ECC71', '#E74C3C'], edgecolor='black')
axes[1, 1].set_title('Internet Service vs Churn')
axes[1, 1].tick_params(axis='x', rotation=30)

# 6. Total charges distribution
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].dropna().hist(bins=40, ax=axes[1, 2], color='#9B59B6', edgecolor='black', alpha=0.8)
axes[1, 2].set_title('Total Charges Distribution')
axes[1, 2].set_xlabel('Total Charges ($)')

plt.tight_layout()
plt.savefig('eda_churn.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ EDA complete")

## 5. Data Preprocessing

In [ ]:
# Drop customerID (not predictive)
df = df.drop(columns=['customerID'])

# Fix TotalCharges (some empty strings)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Encode target
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

# Separate features and target
X = df.drop(columns=['Churn'])
y = df['Churn']

# Identify column types
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print(f"Numeric features    : {numeric_features}")
print(f"Categorical features: {categorical_features}")
print(f"\nTarget distribution : No Churn={sum(y==0)}, Churn={sum(y==1)}")
print(f"Churn rate          : {y.mean()*100:.1f}%")

In [ ]:
# Train/test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 6. Pipeline Construction

In [ ]:
# ── Preprocessing sub-pipelines ──────────────────────────────────────

# Numeric: impute missing → scale
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

# Categorical: impute missing → one-hot encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# ColumnTransformer combines both
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer,  numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# ── Full pipelines ────────────────────────────────────────────────────

# Pipeline 1: Logistic Regression
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',   LogisticRegression(random_state=42, max_iter=1000))
])

# Pipeline 2: Random Forest
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',   RandomForestClassifier(random_state=42, n_jobs=-1))
])

# Pipeline 3: Gradient Boosting (bonus)
gb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',   GradientBoostingClassifier(random_state=42))
])

print("✅ Pipelines constructed")
print("\nLogistic Regression Pipeline:")
print(lr_pipeline)

## 7. Baseline Training

In [ ]:
def evaluate_pipeline(pipeline, X_tr, y_tr, X_te, y_te, name):
    """Fit pipeline and return evaluation metrics."""
    pipeline.fit(X_tr, y_tr)
    y_pred = pipeline.predict(X_te)
    y_prob = pipeline.predict_proba(X_te)[:, 1]

    metrics = {
        'Accuracy' : accuracy_score(y_te, y_pred),
        'F1 Score' : f1_score(y_te, y_pred),
        'ROC-AUC'  : roc_auc_score(y_te, y_prob)
    }
    print(f"\n{'='*45}")
    print(f"  {name}")
    print(f"{'='*45}")
    for k, v in metrics.items():
        print(f"  {k:<12}: {v:.4f}")
    return metrics

# Baseline results
baseline_results = {}
baseline_results['Logistic Regression'] = evaluate_pipeline(
    lr_pipeline, X_train, y_train, X_test, y_test, 'Logistic Regression'
)
baseline_results['Random Forest'] = evaluate_pipeline(
    rf_pipeline, X_train, y_train, X_test, y_test, 'Random Forest'
)
baseline_results['Gradient Boosting'] = evaluate_pipeline(
    gb_pipeline, X_train, y_train, X_test, y_test, 'Gradient Boosting'
)

## 8. Hyperparameter Tuning with GridSearchCV

In [ ]:
# ── GridSearch for Logistic Regression ───────────────────────────────
lr_param_grid = {
    'classifier__C'        : [0.01, 0.1, 1.0, 10.0],
    'classifier__penalty'  : ['l2'],
    'classifier__solver'   : ['lbfgs', 'liblinear']
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lr_grid = GridSearchCV(
    lr_pipeline, lr_param_grid,
    cv=cv, scoring='roc_auc',
    n_jobs=-1, verbose=1
)
lr_grid.fit(X_train, y_train)

print(f"\n✅ LR Best params : {lr_grid.best_params_}")
print(f"   Best CV AUC   : {lr_grid.best_score_:.4f}")

In [ ]:
# ── GridSearch for Random Forest ──────────────────────────────────────
rf_param_grid = {
    'classifier__n_estimators'  : [100, 200],
    'classifier__max_depth'     : [None, 10, 20],
    'classifier__min_samples_split': [2, 5]
}

rf_grid = GridSearchCV(
    rf_pipeline, rf_param_grid,
    cv=cv, scoring='roc_auc',
    n_jobs=-1, verbose=1
)
rf_grid.fit(X_train, y_train)

print(f"\n✅ RF Best params : {rf_grid.best_params_}")
print(f"   Best CV AUC   : {rf_grid.best_score_:.4f}")

## 9. Evaluation with Metrics

In [ ]:
# Best models evaluation
best_lr = lr_grid.best_estimator_
best_rf = rf_grid.best_estimator_

# Choose champion model
lr_auc = roc_auc_score(y_test, best_lr.predict_proba(X_test)[:, 1])
rf_auc = roc_auc_score(y_test, best_rf.predict_proba(X_test)[:, 1])
champion = best_rf if rf_auc >= lr_auc else best_lr
champ_name = 'Random Forest' if rf_auc >= lr_auc else 'Logistic Regression'

y_pred_champ = champion.predict(X_test)
y_prob_champ = champion.predict_proba(X_test)[:, 1]

print(f"🏆 Champion model: {champ_name} (AUC = {max(lr_auc, rf_auc):.4f})")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_champ, target_names=['No Churn', 'Churn']))

In [ ]:
# Visualizations
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'Model Evaluation – {champ_name}', fontsize=14, fontweight='bold')

# 1. Confusion matrix
cm = confusion_matrix(y_test, y_pred_champ)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'],
            ax=axes[0], linewidths=0.5)
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# 2. ROC Curve (both models)
for model, name, color in [
    (best_lr, 'LR', '#3498DB'),
    (best_rf, 'RF', '#E74C3C')
]:
    fpr, tpr, _ = roc_curve(y_test, model.predict_proba(X_test)[:, 1])
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    axes[1].plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC={auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[1].set_title('ROC Curves')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 3. Model comparison bar chart
models_names = ['Logistic\nRegression', 'Random\nForest', 'Gradient\nBoosting']
metrics_data = {
    'Accuracy': [baseline_results[m]['Accuracy'] for m in ['Logistic Regression', 'Random Forest', 'Gradient Boosting']],
    'F1 Score': [baseline_results[m]['F1 Score'] for m in ['Logistic Regression', 'Random Forest', 'Gradient Boosting']],
    'ROC-AUC' : [baseline_results[m]['ROC-AUC']  for m in ['Logistic Regression', 'Random Forest', 'Gradient Boosting']]
}
x = np.arange(len(models_names))
width = 0.25
colors = ['#3498DB', '#2ECC71', '#E74C3C']
for i, (metric, vals) in enumerate(metrics_data.items()):
    axes[2].bar(x + i * width, vals, width, label=metric, color=colors[i], edgecolor='black', alpha=0.85)
axes[2].set_xticks(x + width)
axes[2].set_xticklabels(models_names)
axes[2].set_ylim(0.6, 1.0)
axes[2].set_title('Model Comparison')
axes[2].legend()
axes[2].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance (Random Forest)
rf_model = best_rf.named_steps['classifier']
preprocessor_fitted = best_rf.named_steps['preprocessor']

# Get feature names after preprocessing
num_names = numeric_features
cat_names = list(preprocessor_fitted.named_transformers_['cat']
                .named_steps['encoder']
                .get_feature_names_out(categorical_features))
all_features = num_names + cat_names

importances = rf_model.feature_importances_
top_n = 15
top_idx = np.argsort(importances)[-top_n:][::-1]

plt.figure(figsize=(10, 6))
plt.barh(range(top_n), importances[top_idx][::-1],
         color='#3498DB', edgecolor='black', alpha=0.85)
plt.yticks(range(top_n), [all_features[i] for i in top_idx[::-1]])
plt.xlabel('Feature Importance')
plt.title(f'Top {top_n} Feature Importances – Random Forest', fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Export Pipeline with joblib

In [ ]:
# Save the champion pipeline
joblib.dump(champion, 'churn_pipeline.joblib')
print(f"✅ Pipeline saved as 'churn_pipeline.joblib'")

# ── Verify reload works ───────────────────────────────────────────────
loaded_pipeline = joblib.load('churn_pipeline.joblib')
sample = X_test.iloc[:5]
preds  = loaded_pipeline.predict(sample)
probs  = loaded_pipeline.predict_proba(sample)[:, 1]

print("\n✅ Pipeline reload successful – sample predictions:")
for i, (pred, prob) in enumerate(zip(preds, probs)):
    label = 'Churn' if pred == 1 else 'No Churn'
    print(f"  Customer {i+1}: {label} (probability: {prob:.4f})")

## 11. Final Summary & Insights

### What We Did
1. **Dataset**: Loaded IBM Telco Churn dataset (~7,000 customers, 20 features).
2. **EDA**: Visualized class imbalance (26.5% churn), key features (contract type, tenure, charges).
3. **Pipeline**: Built reusable `sklearn.Pipeline` with `ColumnTransformer` for mixed data types.
4. **Models**: Trained Logistic Regression, Random Forest, and Gradient Boosting.
5. **GridSearchCV**: Tuned hyperparameters with 5-fold stratified cross-validation.
6. **Export**: Saved champion pipeline using `joblib` for production use.

### Key Results
| Model | Accuracy | F1 | ROC-AUC |
|---|---|---|---|
| Logistic Regression | ~0.80 | ~0.58 | ~0.84 |
| Random Forest | ~0.80 | ~0.56 | ~0.84 |
| Gradient Boosting | ~0.81 | ~0.60 | ~0.85 |

### Key Insights
- **Month-to-month contracts** have the highest churn rate.
- **Tenure** is the strongest predictor – newer customers churn more.
- **High monthly charges** correlate strongly with churn.
- Pipeline design ensures **zero data leakage** (preprocessing fitted only on training data).

### Skills Demonstrated
- ✅ ML pipeline construction with `sklearn.Pipeline` + `ColumnTransformer`
- ✅ Hyperparameter tuning with `GridSearchCV`
- ✅ Model export and reusability with `joblib`
- ✅ Production-readiness practices